In [ ]:
# Preprocess OriginalData.xlsx in the current folder and write preprocessed_data.csv

import os
import pandas as pd

INPUT_FILE = "Data/OriginalData.xlsx"
OUTPUT_FILE = "Data/preprocessed_data.csv"

REQUIRED_COLUMNS = [
    "Clone name",
    "HIC retention time (min)",
    "VH Protein",
    "VL Protein",
]

# --- Checks ---
if not os.path.isfile(INPUT_FILE):
    raise FileNotFoundError(
        f"'{INPUT_FILE}' not found in the current folder: {os.getcwd()}"
    )

# --- Load Excel (first sheet by default) ---
# Using openpyxl explicitly for .xlsx files
df = pd.read_excel(INPUT_FILE, sheet_name=0, engine="openpyxl")

# --- Normalize column names (trim spaces) ---
df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]

# --- Validate required columns ---
missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    available = ", ".join(map(str, df.columns))
    needed = ", ".join(missing)
    raise ValueError(
        f"Missing required column(s): {needed}\n"
        f"Available columns: {available}"
    )

# --- Select and filter ---
df = df[REQUIRED_COLUMNS].copy()

time_col = "HIC retention time (min)"
# Convert to numeric; non-numeric becomes NaN
df[time_col] = pd.to_numeric(df[time_col], errors="coerce")

# Keep only rows where the value is numeric (i.e., not NaN)
df = df[df[time_col].notna()].reset_index(drop=True)

# --- Save to CSV ---
df.to_csv(OUTPUT_FILE, index=False)

# --- Simple report ---
print(f"Preprocessing complete. Wrote {len(df)} rows and {df.shape[1]} columns to '{OUTPUT_FILE}'.")
display(df.head())